In [9]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

def texto_elemento(elemento):
    if elemento is None:
        return ''
    return ' '.join(''.join(elemento.itertext()).split())

def minerar_patentes_inpi(diretorio_xml):
    dados_patentes = []

    for arquivo in os.listdir(diretorio_xml):
        if not arquivo.lower().endswith('.xml'):
            continue

        caminho_completo = os.path.join(diretorio_xml, arquivo)

        try:
            tree = ET.parse(caminho_completo)
            root = tree.getroot()

            for processo in root.findall('.//processo'):
                titulares = [
                    titular.get('nome-razao-social', '').strip()
                    for titular in processo.findall('./titulares/titular')
                    if titular.get('nome-razao-social')
                ]
                despachos = [
                    f"{despacho.get('codigo', '').strip()}: {despacho.get('nome', '').strip()}"
                    for despacho in processo.findall('./despachos/despacho')
                ]
                textos_complementares = [
                    texto_elemento(elemento)
                    for elemento in processo.findall('./despachos/despacho/texto-complementar')
                ]
                marcas = processo.findall('./marca')
                especificacoes = [
                    texto_elemento(elemento)
                    for elemento in processo.findall('.//especificacao')
                ]

                dados_patentes.append({
                    'numero_processo': processo.get('numero', '').strip(),
                    'titulares': '; '.join(titulares),
                    'procurador': texto_elemento(processo.find('./procurador')),
                    'nome_marca': '; '.join(
                        texto_elemento(marca.find('./nome')) for marca in marcas
                    ),
                    'apresentacao_marca': '; '.join(
                        marca.get('apresentacao', '').strip() for marca in marcas
                        if marca.get('apresentacao')
                    ),
                    'natureza_marca': '; '.join(
                        marca.get('natureza', '').strip() for marca in marcas
                        if marca.get('natureza')
                    ),
                    'descricao_produtos_servicos': '; '.join(
                        item for item in especificacoes if item
                    ),
                    'texto_complementar': '; '.join(
                        item for item in textos_complementares if item
                    ),
                    'despachos': '; '.join(despachos),
                    'arquivo_origem': arquivo
                })

        except ET.ParseError:
            print(f"Erro estrutural ao analisar o arquivo: {arquivo}")

    return pd.DataFrame(dados_patentes)

# --- Execução ---
pasta_arquivos = './xml_inpi'
df_base_inpi = minerar_patentes_inpi(pasta_arquivos)

print(f'Total de processos: {len(df_base_inpi)}')
display(df_base_inpi.head())

Total de processos: 37393


,numero_processo,titulares,procurador,nome_marca,apresentacao_marca,natureza_marca,descricao_produtos_servicos,texto_complementar,despachos,arquivo_origem
0,906428203,Mercilo João Rigon,,,,,,,IPAS161: Extinção de registro pela expiração d...,RM2907.xml
1,901059528,BUTTER MANAGEMENT LLC,MARIA ELISA SANTUCCI BREVES,,,,,,IPAS161: Extinção de registro pela expiração d...,RM2907.xml
2,906231876,RUBBERMAID INCORPORATED,GUERRA PROPRIEDADE INDUSTRIAL,,,,,,IPAS161: Extinção de registro pela expiração d...,RM2907.xml
3,906204135,QUALIPLAN SERVIÇOS EMPRESARIAIS E SAÚDE EIRELI,BEERRE ASSESSORIA EMPRESARIAL SC LTDA,,,,,,IPAS161: Extinção de registro pela expiração d...,RM2907.xml
4,903083400,ACOPLAN - ASSOCIAÇÃO DOS CORRETORES DE PLANOS ...,ROMEU GUILHERME TRAGANTE,,,,,,IPAS161: Extinção de registro pela expiração d...,RM2907.xml


In [11]:
df_base_inpi.info()

<class 'pandas.DataFrame'>
RangeIndex: 37393 entries, 0 to 37392
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   numero_processo              37393 non-null  str  
 1   titulares                    37393 non-null  str  
 2   procurador                   37393 non-null  str  
 3   nome_marca                   37393 non-null  str  
 4   apresentacao_marca           37393 non-null  str  
 5   natureza_marca               37393 non-null  str  
 6   descricao_produtos_servicos  37393 non-null  str  
 7   texto_complementar           37393 non-null  str  
 8   despachos                    37393 non-null  str  
 9   arquivo_origem               37393 non-null  str  
dtypes: str(10)
memory usage: 34.4 MB


In [13]:
df_base_inpi.to_csv('patentes-br-iti-tentativa.csv', index=False, sep=',', encoding='utf-8')

In [6]:
import os
import epo_ops

client = epo_ops.Client(
    key=os.environ["FnSgDCHCprv64JQkbkxa1bJG93IyHZkaYZT60JQrGVamaKeZ"],
    secret=os.environ["26GbQjjRTfATWbyUytIUlbJVVCFFGNsjsKx1Lpe8GYoGkLTJCJUqPvQLmSGlBuKx"]
)

referencia = epo_ops.models.Docdb("8000056", "BR", "U2")

for endpoint in ("biblio", "abstract"):
    try:
        response = client.published_data(
            reference_type="publication",
            input=referencia,
            endpoint=endpoint
        )
        print(f"[{endpoint}]\n{response.text}")
    except Exception as error:
        print(
            f"Não há dados OPS para BR8000056U2 no endpoint "
            f"'{endpoint}': {error}"
        )

try:
    response = client.published_data(
        reference_type="publication",
        input=referencia,
        endpoint="description"
    )
    print(f"[description]\n{response.text}")
except Exception as error:
    print(f"Descrição não disponível no OPS: {error}")

KeyError: 'FnSgDCHCprv64JQkbkxa1bJG93IyHZkaYZT60JQrGVamaKeZ'

In [ ]:
import requests
import time
from pathlib import Path

# 1. Buscar lista de patentes pelo TXT da RPI (que você já tem acesso)
# 2. Para cada número de pedido, buscar no pePI com sessão autenticada

session = requests.Session()

# Login no pePI
login_url = "https://gru.inpi.gov.br/pePI/servlet/LoginController"
session.post(login_url, data={
    "action": "login",
    "loginInput": "SEU_LOGIN",
    "senhaInput": "SUA_SENHA"
})

# Buscar patente individual
def buscar_patente(numero_pedido):
    url = "https://gru.inpi.gov.br/pePI/servlet/PatentesSearchBasico"
    r = session.get(url, params={"NumberField": numero_pedido})
    return r.text

# Salvar resultado
resultado = buscar_patente("BR102010000001")
Path(f"patentes/{numero_pedido}.html").write_text(resultado)
time.sleep(1)  # respeitar rate limit